# Thí nghiệm Big Data: Multimodal Fake News Detection (Fakeddit Subset)
Notebook này được thiết kế tự động để chạy trên Google Colab nhằm chứng minh năng lực của mô hình Cross-Attention trên tập dữ liệu lớn (Big Data).

## Bước 1: Khởi tạo môi trường và Clone mã nguồn
Tự động xóa thư mục cũ (nếu có) trước khi clone để tránh lỗi khi chạy lại.

In [ ]:
# Xóa thư mục cũ nếu đã tồn tại để tránh lỗi git clone
!rm -rf multimodal-fake-news-detection
print("Đang clone mã nguồn từ GitHub...")
!git clone https://github.com/btsuu25-dev/multimodal-fake-news-detection.git
%cd multimodal-fake-news-detection
print("Đang cài đặt thư viện (mất khoảng 15-20 phút lần đầu)...")
!pip install -r requirements.txt -q
!pip install datasets tqdm requests pandas -q
print("✅ Bước 1 hoàn tất!")

## Bước 1.5: Tối ưu hóa hiệu suất (Ép xung GPU)
Tự động tăng `BATCH_SIZE` lên 64 và `NUM_WORKERS` lên 4 để tận dụng tối đa GPU 16GB VRAM của Colab T4.

> **Lưu ý:** Dùng Regex để thay đúng tên biến hằng số `BATCH_SIZE` và `NUM_WORKERS` (viết hoa) trong file nguồn.

In [ ]:
import os, re

files_to_edit = ['src/train.py', 'src/train_baseline.py']

for file_path in files_to_edit:
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Dùng Regex thay đúng tên biến IN HOA (BATCH_SIZE, NUM_WORKERS)
        content = re.sub(r'BATCH_SIZE\s*=\s*\d+', 'BATCH_SIZE  = 64', content)
        content = re.sub(r'NUM_WORKERS\s*=\s*\d+', 'NUM_WORKERS  = 4', content)
        
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(content)
        print(f'✅ Đã ép xung: {file_path}')
    else:
        print(f'❌ Không tìm thấy file: {file_path}')

print('\n🚀 Hoàn tất! BATCH_SIZE=64 và NUM_WORKERS=4 đã được áp dụng cho cả 2 file train!')

## Bước 2: Tải và Tiền xử lý dữ liệu Fakeddit
Dùng đa luồng tải ảnh từ 80.000 link để lọc ra chính xác 30.000 ảnh sống sót.

In [ ]:
import os
import requests
import pandas as pd
from datasets import load_dataset
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

print("Đang tải dữ liệu gốc Fakeddit từ máy chủ Parquet tốc độ cao...")

dataset = load_dataset('AdoCleanCode/Fakeddit', split='train', streaming=False)
df = dataset.to_pandas()

df = df[df['hasImage'] == True]
df['2_way_label'] = df['label_name'].apply(lambda x: 0 if x == 'True' else 1)

# Lấy 80.000 link để bù trừ cho tỷ lệ link chết ~50%
df = df[['image_url', 'text', '2_way_label']].dropna().head(80000)

os.makedirs('data/images', exist_ok=True)
os.makedirs('data/splits', exist_ok=True)

valid_rows = []
MAX_SAMPLES = 30000  # Mục tiêu chốt cứng 30.000 mẫu

def download_image(row):
    if len(valid_rows) >= MAX_SAMPLES: 
        return
        
    url = row['image_url']
    img_name = str(url).split('/')[-1].split('?')[0] 
    img_path = os.path.join('data/images', img_name)
    
    if not os.path.exists(img_path):
        try:
            res = requests.get(url, timeout=3)
            if res.status_code == 200:
                with open(img_path, 'wb') as f:
                    f.write(res.content)
                if len(valid_rows) < MAX_SAMPLES:
                    valid_rows.append({
                        'image_path': img_path,
                        'text': row['text'],
                        'label': row['2_way_label']
                    })
        except:
            pass

print(f"Bắt đầu quét qua 80.000 link để chắt lọc đúng {MAX_SAMPLES} ảnh sống...")
rows_list = [row for _, row in df.iterrows()]
with ThreadPoolExecutor(max_workers=32) as executor:
    list(tqdm(executor.map(download_image, rows_list), total=len(rows_list)))

valid_rows = valid_rows[:MAX_SAMPLES]
print(f"\nĐã tải và chắt lọc thành công CHÍNH XÁC {len(valid_rows)} mẫu dữ liệu Fakeddit gốc hợp lệ!")

## Bước 2.5: Lọc ảnh bị hỏng (Corrupt Filter)
Kiểm tra và loại bỏ các ảnh bị hỏng file trước khi lưu vào CSV để đảm bảo 100% dữ liệu train là ảnh thật.

In [ ]:
from PIL import Image

print("Đang kiểm tra tính toàn vẹn của ảnh...")

def is_valid_image(path):
    try:
        img = Image.open(path)
        img.verify()  # Kiểm tra file ảnh không bị hỏng
        return True
    except:
        return False

before = len(valid_rows)
valid_rows = [row for row in valid_rows if is_valid_image(row['image_path'])]
after = len(valid_rows)

print(f"Giữ lại {after}/{before} ảnh hợp lệ (đã lọc bỏ {before - after} ảnh bị hỏng)")
print("✅ Dữ liệu đã được làm sạch 100%!")

## Bước 3: Chia tập dữ liệu (Train/Val/Test)

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

final_df = pd.DataFrame(valid_rows)

# Chia theo tỷ lệ 80-10-10
train_df, temp_df = train_test_split(final_df, test_size=0.2, random_state=42, stratify=final_df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

train_df.to_csv('data/splits/train.csv', index=False)
val_df.to_csv('data/splits/val.csv', index=False)
test_df.to_csv('data/splits/test.csv', index=False)

print("Đã lưu các file phân chia dữ liệu vào thư mục data/splits/")
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

## Bước 4: Huấn luyện Mô hình Concat (Baseline)
Cùng xem sức mạnh của kiến trúc đơn giản khi gặp Big Data.

In [ ]:
import os
os.environ['PYTHONPATH'] = '.'
!python src/train_baseline.py

## Bước 5: Huấn luyện Mô hình Cross-Attention (Mô hình chính)
Kỳ vọng: Cross-Attention sẽ phát huy sức mạnh bắt chéo đặc trưng trên tập dữ liệu đa dạng và lớn hơn này.

In [ ]:
import os
os.environ['PYTHONPATH'] = '.'
!python src/train.py

## Bước 6: Đánh giá và Xuất báo cáo (HTML)

In [ ]:
!python src/evaluate.py
!python src/evaluation/html_report.py
print("Hoàn tất! Bạn có thể tải file results/dashboard.html về máy tính để xem kết quả so sánh cuối cùng!")